# Лабораторная работа: PyTorch Shapes, Dot Products и Mini-Attention

Цель: быстро и уверенно освоить PyTorch-операторы, которые нужны для Transformer-лабораторных:

- `shape`;
- `view` / `reshape`;
- `unsqueeze`;
- broadcasting;
- transpose / `.T`;
- `@` как matrix multiplication;
- `x @ x.T` как pairwise dot products;
- `softmax`;
- mini scaled dot-product attention;
- batched attention;
- causal masks;
- multi-head attention shapes.

Эта лабораторная отдельная и подготовительная: после нее positional embeddings и attention будут восприниматься гораздо легче.

## Learning outcomes

После выполнения лабораторной студент должен уметь:

- читать и предсказывать формы тензоров;
- безопасно менять форму через `view` / `reshape`;
- объяснять, когда нужен `unsqueeze`;
- отличать elementwise product от dot product;
- объяснять, почему `x @ x.T` дает матрицу попарных dot products;
- реализовать mini scaled dot-product attention руками;
- писать batched attention с mask;
- уверенно переходить между `(B, T, D_model)` и `(B, H, T, D_head)`.

## 0. Подготовка

In [ ]:
from __future__ import annotations

import math

import torch
from torch import Tensor
from torch.nn import functional as F


torch.manual_seed(7)
torch.set_printoptions(precision=4, sci_mode=False)

print("PyTorch:", torch.__version__)


def report_check(name: str, value: float, threshold: float | None = None) -> None:
    """Печатает метрику проверки и статус OK/FAIL, если задан порог."""
    if threshold is None:
        print(f"{name}: {value:.4f}")
    else:
        status = "OK" if value <= threshold else "FAIL"
        print(f"{name}: {value:.2e} <= {threshold:.2e} [{status}]")


def preview_tensor(name: str, tensor: Tensor, rows: int = 4, cols: int = 8) -> None:
    """Печатает shape и небольшой фрагмент тензора."""
    print(f"{name} shape: {tuple(tensor.shape)}")
    if tensor.ndim == 1:
        print(tensor[:cols])
    elif tensor.ndim == 2:
        print(tensor[:rows, :cols])
    else:
        print(tensor)


def print_square_matrix(name: str, matrix: Tensor, labels: list[str] | None = None) -> None:
    """Печатает квадратную матрицу, опционально с подписями строк/столбцов."""
    print(name)
    if labels is None:
        print(matrix)
        return
    header = "      " + " ".join(f"{label[:4]:>7}" for label in labels)
    print(header)
    for label, row in zip(labels, matrix):
        values = " ".join(f"{value.item():7.2f}" for value in row)
        print(f"{label[:4]:>4}  {values}")


## 1. Shapes: читать тензор как таблицу размерностей

В PyTorch почти каждая ошибка в Transformer-коде начинается с неправильной формы.

Будем использовать соглашение:

```text
B = batch size
T = sequence length
D = embedding dimension
```

Тензор token embeddings обычно имеет форму `(B, T, D)`.

In [ ]:
B = 2
T = 4
D = 3

x = torch.arange(B * T * D, dtype=torch.float32).view(B, T, D)
preview_tensor("x", x)
print("x.shape:", x.shape)
print("batch 0 shape:", x[0].shape)
print("token 0 in batch 0 shape:", x[0, 0].shape)


### Мини-лабораторная 1: shape reading

Создайте:

- `batch0`: первый элемент batch, форма `(T, D)`;
- `token2`: третий токен первого batch, форма `(D,)`;
- `feature_column`: все batch-и и все токены, но только feature с индексом `1`, форма `(B, T)`.

In [ ]:
# Мини-лабораторная 1
batch0 = None
token2 = None
feature_column = None

# здесь ваш код


In [ ]:
# Проверка мини-лабораторной 1
assert tuple(batch0.shape) == (T, D)
assert tuple(token2.shape) == (D,)
assert tuple(feature_column.shape) == (B, T)
assert torch.allclose(batch0, x[0])
assert torch.allclose(token2, x[0, 2])
assert torch.allclose(feature_column, x[:, :, 1])
print("shape reading: OK")


## 2. `view` и `reshape`: та же память, другая форма

`view` меняет форму тензора без изменения данных. Главное правило: число элементов должно сохраняться.

```python
x.numel() == reshaped.numel()
```

Для flatten batch и sequence часто делают:

```text
(B, T, D) -> (B*T, D)
```

### Мини-лабораторная 2: flatten и restore

Реализуйте две функции:

- `flatten_batch_tokens(x)`: `(B, T, D) -> (B*T, D)`;
- `restore_batch_tokens(x, batch_size, seq_len)`: `(B*T, D) -> (B, T, D)`.

In [ ]:
# Мини-лабораторная 2

def flatten_batch_tokens(x: Tensor) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


def restore_batch_tokens(x: Tensor, batch_size: int, seq_len: int) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


In [ ]:
# Проверка мини-лабораторной 2
flat = flatten_batch_tokens(x)
restored = restore_batch_tokens(flat, batch_size=B, seq_len=T)

assert tuple(flat.shape) == (B * T, D)
assert tuple(restored.shape) == tuple(x.shape)
assert torch.allclose(restored, x)
print("flatten/restore: OK")


## 3. `unsqueeze` и broadcasting

`unsqueeze(dim)` добавляет размерность длины 1. Это часто нужно, чтобы PyTorch мог автоматически применить broadcasting.

Пример для positional vectors:

```text
position_vectors: (T, D)
embeddings:       (B, T, D)
position_vectors.unsqueeze(0): (1, T, D)
```

После этого можно сложить `(B, T, D) + (1, T, D)`.

### Мини-лабораторная 3: добавьте позиционный bias через broadcasting

Создайте `position_bias` формы `(T, D)`, где каждая строка заполнена номером позиции. Затем прибавьте его к `x`.

In [ ]:
# Мини-лабораторная 3
position_bias = None
x_with_position_bias = None

# здесь ваш код


In [ ]:
# Проверка мини-лабораторной 3
assert tuple(position_bias.shape) == (T, D)
assert tuple(x_with_position_bias.shape) == tuple(x.shape)
assert torch.allclose(position_bias[0], torch.zeros(D))
assert torch.allclose(position_bias[-1], torch.full((D,), T - 1, dtype=position_bias.dtype))
expected = x + position_bias.unsqueeze(0)
assert torch.allclose(x_with_position_bias, expected)
print("broadcast position bias: OK")


## 4. `@`, transpose и dot product

Для двух векторов `a` и `b` dot product:

```python
(a * b).sum()
```

Для матрицы `X` формы `(T, D)` выражение:

```python
X @ X.T
```

возвращает матрицу формы `(T, T)`, где элемент `[i, j]` равен dot product токенов `i` и `j`.

In [ ]:
X = x[0]  # (T, D)
print("X shape:", X.shape)
print("X.T shape:", X.T.shape)
print("X @ X.T shape:", (X @ X.T).shape)


### Мини-лабораторная 4: pairwise dot products

Реализуйте `pairwise_dot_products(X)`. Нельзя использовать циклы: используйте `@` и `.T`.

In [ ]:
# Мини-лабораторная 4

def pairwise_dot_products(X: Tensor) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


In [ ]:
# Проверка мини-лабораторной 4
dots = pairwise_dot_products(X)
assert tuple(dots.shape) == (T, T)
assert torch.allclose(dots, dots.T)
assert torch.allclose(torch.diag(dots), (X * X).sum(dim=-1))
manual_0_1 = (X[0] * X[1]).sum().item()
assert abs(dots[0, 1].item() - manual_0_1) < 1e-7
print_square_matrix("pairwise dot products", dots)


## 5. Softmax: scores -> weights

Attention scores сами по себе могут быть любыми числами. `softmax` превращает каждую строку scores в распределение:

- все значения неотрицательны;
- сумма строки равна 1.

В attention обычно применяют `F.softmax(scores, dim=-1)`.

### Мини-лабораторная 5: row-wise softmax

Реализуйте `row_softmax(scores)`.

In [ ]:
# Мини-лабораторная 5

def row_softmax(scores: Tensor) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


In [ ]:
# Проверка мини-лабораторной 5
scores = torch.tensor([[1.0, 2.0, 3.0], [2.0, 0.0, -2.0]])
weights = row_softmax(scores)
assert tuple(weights.shape) == tuple(scores.shape)
assert torch.all(weights >= 0)
row_sum_error = (weights.sum(dim=-1) - 1.0).abs().max().item()
report_check("row sum error", row_sum_error, 1e-7)
assert row_sum_error < 1e-7
assert weights[0, 2] > weights[0, 1] > weights[0, 0]
print(weights)


## 6. Mini scaled dot-product attention

Теперь соберем все вместе:

```text
Q = X W_q
K = X W_k
V = X W_v
scores = Q K^T / sqrt(d_k)
attn = softmax(scores)
out = attn V
```

Здесь `Q @ K.T` похож на `X @ X.T`, но сравниваются не исходные токены, а query/key-представления.

In [ ]:
torch.manual_seed(23)
T = 5
D = 8
D_k = 4

X = torch.randn(T, D)
W_q = torch.randn(D, D_k) / math.sqrt(D)
W_k = torch.randn(D, D_k) / math.sqrt(D)
W_v = torch.randn(D, D_k) / math.sqrt(D)

print("X:", X.shape)
print("W_q:", W_q.shape)


### Мини-лабораторная 6: реализуйте mini_attention

Функция должна вернуть:

```python
out, attn, scores = mini_attention(X, W_q, W_k, W_v)
```

Используйте `@`, `.T`, `math.sqrt`, `F.softmax`.

In [ ]:
# Мини-лабораторная 6

def mini_attention(
    X: Tensor,
    W_q: Tensor,
    W_k: Tensor,
    W_v: Tensor,
) -> tuple[Tensor, Tensor, Tensor]:
    # здесь ваш код
    raise NotImplementedError


In [ ]:
# Проверка мини-лабораторной 6
out, attn, scores = mini_attention(X, W_q, W_k, W_v)
assert tuple(scores.shape) == (T, T)
assert tuple(attn.shape) == (T, T)
assert tuple(out.shape) == (T, D_k)

row_sum_error = (attn.sum(dim=-1) - 1.0).abs().max().item()
report_check("attention row-sum error", row_sum_error, 1e-6)
assert row_sum_error < 1e-6
assert torch.all(attn >= 0)

Q = X @ W_q
K = X @ W_k
V = X @ W_v
manual_score_0_1 = (Q[0] * K[1]).sum().item() / math.sqrt(D_k)
score_error = abs(scores[0, 1].item() - manual_score_0_1)
report_check("manual score[0,1] error", score_error, 1e-7)
assert score_error < 1e-7

manual_out_0 = (attn[0].unsqueeze(1) * V).sum(dim=0)
out_error = (out[0] - manual_out_0).abs().max().item()
report_check("manual out[0] error", out_error, 1e-7)
assert out_error < 1e-7

preview_tensor("attention weights", attn, rows=T, cols=T)


## 7. Shape gym: индексация, `keepdim`, быстрые срезы

Эта секция специально сделана как набор коротких повторений. Цель: не просто понять, а набить руку на чтении `(B, T, D)` и сохранении нужных размерностей.

Правило тренировки: сначала вслух предскажите shape, потом пишите одну-две строки PyTorch, потом запускайте проверку.


In [ ]:
torch.manual_seed(101)
B_g = 3
T_g = 5
D_g = 4

xg = torch.arange(B_g * T_g * D_g, dtype=torch.float32).view(B_g, T_g, D_g)
preview_tensor("xg", xg)
print("B_g, T_g, D_g:", B_g, T_g, D_g)


### Мини-лабораторная 7: shape gym

Создайте:

- `last_token`: последний токен каждого элемента batch, shape `(B_g, D_g)`;
- `first_two_tokens`: первые два токена каждого элемента batch, shape `(B_g, 2, D_g)`;
- `feature_1_to_2`: features с индексами `1` и `2`, shape `(B_g, T_g, 2)`;
- `token_norms_keepdim`: L2-норма каждого токена, но с сохранением feature-оси, shape `(B_g, T_g, 1)`;
- `centered_over_time`: вычтите средний токен внутри каждого batch по оси `T`, shape `(B_g, T_g, D_g)`.


In [ ]:
# Мини-лабораторная 7
last_token = None
first_two_tokens = None
feature_1_to_2 = None
token_norms_keepdim = None
centered_over_time = None

# здесь ваш код


In [ ]:
# Проверка мини-лабораторной 7
assert tuple(last_token.shape) == (B_g, D_g)
assert tuple(first_two_tokens.shape) == (B_g, 2, D_g)
assert tuple(feature_1_to_2.shape) == (B_g, T_g, 2)
assert tuple(token_norms_keepdim.shape) == (B_g, T_g, 1)
assert tuple(centered_over_time.shape) == (B_g, T_g, D_g)

assert torch.allclose(last_token, xg[:, -1, :])
assert torch.allclose(first_two_tokens, xg[:, :2, :])
assert torch.allclose(feature_1_to_2, xg[:, :, 1:3])
assert torch.allclose(token_norms_keepdim.squeeze(-1), xg.norm(dim=-1))
assert torch.allclose(centered_over_time.mean(dim=1), torch.zeros(B_g, D_g))
print("shape gym: OK")


## 8. Broadcasting drills: вставить оси без паники

В Transformer-коде часто нужно добавить bias или scale с формой `(T,)`, `(D,)` или `(B,)` к тензору `(B, T, D)`. Почти всегда это решается через `unsqueeze`.


### Мини-лабораторная 8: три разных broadcast

Даны:

- `token_scale`: shape `(T_g,)`;
- `feature_shift`: shape `(D_g,)`;
- `batch_shift`: shape `(B_g,)`.

Создайте:

- `token_scale_grid`: shape `(1, T_g, 1)`;
- `feature_shift_grid`: shape `(1, 1, D_g)`;
- `batch_shift_grid`: shape `(B_g, 1, 1)`;
- `broadcast_mix = xg * token_scale_grid + feature_shift_grid + batch_shift_grid`, shape `(B_g, T_g, D_g)`.


In [ ]:
# Мини-лабораторная 8
token_scale = torch.linspace(1.0, 2.0, T_g)
feature_shift = torch.arange(D_g, dtype=torch.float32)
batch_shift = torch.arange(B_g, dtype=torch.float32) * 10.0

token_scale_grid = None
feature_shift_grid = None
batch_shift_grid = None
broadcast_mix = None

# здесь ваш код


In [ ]:
# Проверка мини-лабораторной 8
assert tuple(token_scale_grid.shape) == (1, T_g, 1)
assert tuple(feature_shift_grid.shape) == (1, 1, D_g)
assert tuple(batch_shift_grid.shape) == (B_g, 1, 1)
assert tuple(broadcast_mix.shape) == (B_g, T_g, D_g)

manual = xg * token_scale.view(1, T_g, 1) + feature_shift.view(1, 1, D_g) + batch_shift.view(B_g, 1, 1)
assert torch.allclose(broadcast_mix, manual)
assert torch.allclose(broadcast_mix[2, 3], xg[2, 3] * token_scale[3] + feature_shift + batch_shift[2])
print("broadcast drills: OK")


## 9. Batched matrix multiplication

Для одного sequence было `X @ X.T -> (T, T)`. Для batch почти то же самое:

```text
(B, T, D) @ (B, D, T) -> (B, T, T)
```

Только вместо `.T` используйте `transpose(-1, -2)`, потому что `.T` удобен только для 2D.


### Мини-лабораторная 9: batch pairwise dots и linear

Реализуйте:

- `batched_pairwise_dot_products(x)`: `(B, T, D) -> (B, T, T)`;
- `batched_linear(x, W)`: `(B, T, D) @ (D, M) -> (B, T, M)`.


In [ ]:
# Мини-лабораторная 9
torch.manual_seed(202)
xb = torch.randn(2, 4, 3)
W_small = torch.randn(3, 5)


def batched_pairwise_dot_products(x: Tensor) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


def batched_linear(x: Tensor, W: Tensor) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


In [ ]:
# Проверка мини-лабораторной 9
batch_dots = batched_pairwise_dot_products(xb)
projected = batched_linear(xb, W_small)

assert tuple(batch_dots.shape) == (2, 4, 4)
assert tuple(projected.shape) == (2, 4, 5)
assert torch.allclose(batch_dots[0], xb[0] @ xb[0].T)
assert torch.allclose(batch_dots[1, 2, 3], (xb[1, 2] * xb[1, 3]).sum())
assert torch.allclose(projected[0, 1], xb[0, 1] @ W_small)
print("batched matmul: OK")


## 10. Q/K/V для batch

Теперь переходим от одного `X: (T, D)` к обычному Transformer-входу `X: (B, T, D_model)`.

Проекции `W_q`, `W_k`, `W_v` применяются к последней размерности.


### Мини-лабораторная 10: batched QKV projections

Реализуйте `project_qkv_batch(X, W_q, W_k, W_v)`, которая возвращает `Q, K, V`.

Ожидаемые формы:

```text
X:   (B_att, T_att, D_model)
W_q: (D_model, D_k)
Q:   (B_att, T_att, D_k)
```


In [ ]:
# Мини-лабораторная 10
torch.manual_seed(303)
B_att = 2
T_att = 4
D_model = 6
D_k = 3

X_att = torch.randn(B_att, T_att, D_model)
W_q_batch = torch.randn(D_model, D_k) / math.sqrt(D_model)
W_k_batch = torch.randn(D_model, D_k) / math.sqrt(D_model)
W_v_batch = torch.randn(D_model, D_k) / math.sqrt(D_model)


def project_qkv_batch(
    X: Tensor,
    W_q: Tensor,
    W_k: Tensor,
    W_v: Tensor,
) -> tuple[Tensor, Tensor, Tensor]:
    # здесь ваш код
    raise NotImplementedError


In [ ]:
# Проверка мини-лабораторной 10
Q_b, K_b, V_b = project_qkv_batch(X_att, W_q_batch, W_k_batch, W_v_batch)

assert tuple(Q_b.shape) == (B_att, T_att, D_k)
assert tuple(K_b.shape) == (B_att, T_att, D_k)
assert tuple(V_b.shape) == (B_att, T_att, D_k)
assert torch.allclose(Q_b[0, 0], X_att[0, 0] @ W_q_batch)
assert torch.allclose(K_b[1, 2], X_att[1, 2] @ W_k_batch)
assert torch.allclose(V_b, X_att @ W_v_batch)
print("batched QKV projections: OK")


## 11. Causal mask: запрет смотреть в будущее

В language modeling токен на позиции `i` не должен видеть позиции `j > i`.

В этой лабораторной договоримся: `True` в mask значит "запретить attention на эту позицию".


### Мини-лабораторная 11: causal mask и `masked_fill`

Реализуйте:

- `make_causal_mask(seq_len)`: возвращает bool-тензор `(seq_len, seq_len)`, где `True` выше диагонали;
- `apply_attention_mask(scores, mask, fill_value=-1e9)`: заменяет запрещенные scores на `fill_value`.


In [ ]:
# Мини-лабораторная 11

def make_causal_mask(seq_len: int) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


def apply_attention_mask(
    scores: Tensor,
    mask: Tensor,
    fill_value: float = -1e9,
) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


In [ ]:
# Проверка мини-лабораторной 11
mask = make_causal_mask(T_att)
toy_scores = torch.arange(T_att * T_att, dtype=torch.float32).view(T_att, T_att)
masked_scores = apply_attention_mask(toy_scores, mask)

assert mask.dtype == torch.bool
assert tuple(mask.shape) == (T_att, T_att)
assert mask[0, 1] and mask[0, -1]
assert not mask[0, 0]
assert not mask[-1, 0]
assert torch.all(mask == torch.triu(torch.ones(T_att, T_att, dtype=torch.bool), diagonal=1))
assert masked_scores[0, 1].item() < -1e8
assert masked_scores[0, 0].item() == toy_scores[0, 0].item()
assert masked_scores[-1, 0].item() == toy_scores[-1, 0].item()
print("causal mask: OK")


## 12. Batched scaled dot-product attention

Теперь собираем batched attention:

```text
Q, K, V: (B, T, D_k)
scores = Q @ K.transpose(-1, -2) / sqrt(D_k)  # (B, T, T)
attn = softmax(scores, dim=-1)
out = attn @ V                                # (B, T, D_k)
```


### Мини-лабораторная 12: batched attention с optional mask

Реализуйте `batched_attention(Q, K, V, mask=None)`.

Требования:

- без mask работает как обычный scaled dot-product attention;
- если `mask` имеет форму `(T, T)`, примените его ко всем batch;
- если `mask` имеет форму `(B, T, T)`, примените свой mask к каждому элементу batch;
- функция возвращает `out, attn, scores`.


In [ ]:
# Мини-лабораторная 12

def batched_attention(
    Q: Tensor,
    K: Tensor,
    V: Tensor,
    mask: Tensor | None = None,
) -> tuple[Tensor, Tensor, Tensor]:
    # здесь ваш код
    raise NotImplementedError


In [ ]:
# Проверка мини-лабораторной 12
causal_mask = make_causal_mask(T_att)
out_b, attn_b, scores_b = batched_attention(Q_b, K_b, V_b, mask=causal_mask)

assert tuple(scores_b.shape) == (B_att, T_att, T_att)
assert tuple(attn_b.shape) == (B_att, T_att, T_att)
assert tuple(out_b.shape) == (B_att, T_att, D_k)

row_sum_error = (attn_b.sum(dim=-1) - 1.0).abs().max().item()
report_check("batched attention row-sum error", row_sum_error, 1e-6)
assert row_sum_error < 1e-6
assert torch.all(attn_b >= 0)
assert attn_b.masked_select(causal_mask.unsqueeze(0)).max().item() < 1e-6

manual_scores_0 = Q_b[0] @ K_b[0].T / math.sqrt(D_k)
manual_scores_0 = manual_scores_0.masked_fill(causal_mask, -1e9)
assert torch.allclose(scores_b[0], manual_scores_0)
assert torch.allclose(out_b[0, 2], attn_b[0, 2] @ V_b[0])
print("batched attention: OK")


## 13. Multi-head shapes: split и combine

Классическая форма для multi-head attention:

```text
(B, T, D_model) -> (B, H, T, D_head)
D_model = H * D_head
```

После `transpose` память часто становится non-contiguous, поэтому при обратном `view` обычно нужен `.contiguous()`.


### Мини-лабораторная 13: split heads и combine heads

Реализуйте:

- `split_heads(x, num_heads)`: `(B, T, D_model) -> (B, H, T, D_head)`;
- `combine_heads(x)`: `(B, H, T, D_head) -> (B, T, H*D_head)`.


In [ ]:
# Мини-лабораторная 13
torch.manual_seed(404)
B_mh = 2
T_mh = 5
D_model_mh = 12
H = 3
D_head = D_model_mh // H

x_mh = torch.randn(B_mh, T_mh, D_model_mh)


def split_heads(x: Tensor, num_heads: int) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


def combine_heads(x: Tensor) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


In [ ]:
# Проверка мини-лабораторной 13
heads = split_heads(x_mh, H)
combined = combine_heads(heads)

assert tuple(heads.shape) == (B_mh, H, T_mh, D_head)
assert tuple(combined.shape) == (B_mh, T_mh, D_model_mh)
assert torch.allclose(combined, x_mh)
assert torch.allclose(heads[:, 0], x_mh[:, :, :D_head])
assert torch.allclose(heads[:, -1], x_mh[:, :, -D_head:])
print("split/combine heads: OK")


## 14. Multi-head attention skeleton

Здесь вся задача почти полностью про shapes. Если предыдущие секции стали автоматическими, этот блок уже ощущается как сборка конструктора.


### Мини-лабораторная 14: полный forward multi-head attention

Реализуйте `multi_head_attention(...)`.

Функция должна:

1. Посчитать `Q, K, V` через матричные проекции.
2. Разбить их на heads.
3. Посчитать scaled dot-product attention отдельно в каждой head.
4. Склеить heads обратно.
5. Применить выходную проекцию `W_o`.

Верните `out, attn`, где `attn` имеет форму `(B, H, T, T)`.


In [ ]:
# Мини-лабораторная 14
torch.manual_seed(505)
W_q_mh = torch.randn(D_model_mh, D_model_mh) / math.sqrt(D_model_mh)
W_k_mh = torch.randn(D_model_mh, D_model_mh) / math.sqrt(D_model_mh)
W_v_mh = torch.randn(D_model_mh, D_model_mh) / math.sqrt(D_model_mh)
W_o_mh = torch.randn(D_model_mh, D_model_mh) / math.sqrt(D_model_mh)
mh_mask = make_causal_mask(T_mh)


def multi_head_attention(
    X: Tensor,
    W_q: Tensor,
    W_k: Tensor,
    W_v: Tensor,
    W_o: Tensor,
    num_heads: int,
    mask: Tensor | None = None,
) -> tuple[Tensor, Tensor]:
    # здесь ваш код
    raise NotImplementedError


In [ ]:
# Проверка мини-лабораторной 14
mh_out, mh_attn = multi_head_attention(
    x_mh,
    W_q_mh,
    W_k_mh,
    W_v_mh,
    W_o_mh,
    num_heads=H,
    mask=mh_mask,
)

assert tuple(mh_out.shape) == (B_mh, T_mh, D_model_mh)
assert tuple(mh_attn.shape) == (B_mh, H, T_mh, T_mh)

row_sum_error = (mh_attn.sum(dim=-1) - 1.0).abs().max().item()
report_check("multi-head attention row-sum error", row_sum_error, 1e-6)
assert row_sum_error < 1e-6
assert torch.all(mh_attn >= 0)
assert mh_attn.masked_select(mh_mask.view(1, 1, T_mh, T_mh)).max().item() < 1e-6
print("multi-head attention: OK")


## 15. Output head: logits по каждому токену

Последний типичный Transformer-паттерн: превратить каждый hidden state в logits по словарю или классам.

```text
hidden: (B, T, D_model)
W_out:  (D_model, C)
logits: (B, T, C)
```


### Мини-лабораторная 15: token classifier logits

Реализуйте `token_classifier_logits(hidden, W, b)`.

Дополнительно создайте `flat_logits`, где batch и time склеены:

```text
logits:      (B, T, C)
flat_logits: (B*T, C)
```


In [ ]:
# Мини-лабораторная 15
torch.manual_seed(606)
C = 7
W_cls = torch.randn(D_model_mh, C) / math.sqrt(D_model_mh)
b_cls = torch.randn(C)


def token_classifier_logits(hidden: Tensor, W: Tensor, b: Tensor) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


logits = None
flat_logits = None

# здесь ваш код


In [ ]:
# Проверка мини-лабораторной 15
assert tuple(logits.shape) == (B_mh, T_mh, C)
assert tuple(flat_logits.shape) == (B_mh * T_mh, C)
assert torch.allclose(logits, token_classifier_logits(x_mh, W_cls, b_cls))
assert torch.allclose(flat_logits, logits.reshape(B_mh * T_mh, C))
assert torch.allclose(logits[0, 0], x_mh[0, 0] @ W_cls + b_cls)
print("token classifier logits: OK")


## 16. Итоговый вызов

Ответьте коротко:

1. Почему `x @ x.T` дает `(T, T)`?
2. Почему для batch нужно `transpose(-1, -2)`, а не просто `.T`?
3. Чем `Q @ K.T` отличается от `X @ X.T`?
4. Почему строки `attn` должны суммироваться в 1?
5. Что делает causal mask и почему `True` удобно трактовать как "запретить"?
6. Как перейти от `(B, T, D_model)` к `(B, H, T, D_head)` и обратно?
7. Где в `out = attn @ V` происходит смешивание информации между токенами?
